# Deploy de modelos

Nesta aula, vamos aprender como fazer o deploy nosso modelo de recomendação.

**Governança via Unity Catalog:** Gerenciamos versões e permissões no catálogo centralizado.
**Aliases (Champion/Challenger):** Em vez de apontar para a "Versão X", o aplicativo aponta para o "Champion". Isso permite trocar o modelo por trás da API sem alterar o código do aplicativo.

In [0]:
%%capture
## instalando os pacotes
%pip install databricks lightgbm
%pip install databricks-feature-engineering
dbutils.library.restartPython()

#Governança com MlflowClient e Deploy Batch

## O que é o `MlflowClient`?

O `MlflowClient` é a API de mais baixo nível do MLflow. Enquanto funções como `mlflow.start_run()` ou `mlflow.log_model()` lidam com o rastreamento automático da célula atual, o `MlflowClient` interage diretamente com o servidor central de metadados do **Unity Catalog**.

Ele é utilizado para ações administrativas e de governança, tais como:

* Consultar versões existentes de um modelo.
* Criar, deletar ou modificar tags e metadados.
* Mapear e gerenciar **Aliases** (como `@Champion` e `@Challenger`) para desacoplar a engenharia de software da engenharia de machine learning.

### Funções Utilizadas

* **`client.get_registered_model(model_name)`**: Acessa o catálogo e traz um objeto contendo os metadados do modelo de três níveis (`catalogo.esquema.modelo`), incluindo quem o criou, quando e a lista de todas as versões salvas.
* **`client.set_registered_model_alias(model_name, alias, version)`**: Vincula uma tag de produção (*Alias*) à versão. Na arquitetura moderna do Unity Catalog, os estágios antigos (`Staging`/`Production`) foram substituídos por Aliases flexíveis.

**Por que usamos Aliases?** O seu pipeline de escoragem em lote ou a sua API REST não vão apontar para a `Versão 4` fixa. Eles vão apontar para `model_name@Champion`. Quando um modelo novo for treinado e validado, basta mover a tag `@Champion` para a nova versão via Client. Todo o seu ecossistema produtivo se atualiza automaticamente sem que uma única linha de código de engenharia precise ser alterada.

### Como o Spark processa isso por baixo do pano?

1. **`mlflow.pyfunc.spark_udf`**: Transforma o artefato do LightGBM/XGBoost em uma função nativa do Spark SQL. Ele lê a **Signature (Assinatura)** do modelo para mapear quais colunas o Spark deve injetar.
2. **Distribuição nos Nós Workers**: O Spark divide a sua tabela Delta de entrada em múltiplas partições. Cada nó *Worker* do cluster recebe um pedaço dos dados e uma cópia isolada do modelo carregado pela UDF.
3. **Escoragem em Paralelo**: Os nós processam o método `.predict_proba()` simultaneamente em memória de forma massiva, gerando a nova coluna de score sem gargalos no nó *Driver*.
4. **Gravação Delta**: O resultado final pode ser salvo de volta no Unity Catalog usando `.write.mode("overwrite").saveAsTable()`.

In [0]:
from mlflow.tracking import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql.functions import col

catalogo = "workspace"
esquema = "gold"
model_name = f"{catalogo}.{esquema}.propensao_compra_modelo"

# Criando o cliente do MLflow
client = MlflowClient()

print(f"Buscando as versões do modelo {model_name} no Unity Catalog...")

# Em vez de get_registered_model, buscamos todas as versões associadas a esse nome
versions = client.search_model_versions(f"name='{model_name}'")

# Ordenamos as versões de forma decrescente para garantir que a mais recente fique no topo
latest_version = sorted([int(v.version) for v in versions], reverse=True)[0]

# Agora definimos o Alias usando a versão dinâmica recuperada
client.set_registered_model_alias(model_name, "Champion", str(latest_version))

print(f" Versão {latest_version} definida como @Champion com sucesso!")